In [ ]:
"""
Inference script for:
  suswitha/GO-Onotology-Term-IDMeta-Llama-3.1-8B-Instruct-Reference

Fine-tuned Llama 3.1-8B-Instruct model for GO (Gene Ontology) Term ID prediction.

Requirements:
    pip install torch transformers accelerate bitsandbytes
"""

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# ── Configuration ──────────────────────────────────────────────────────────────

MODEL_ID = "suswitha/GO-Onotology-Term-IDMeta-Llama-3.1-8B-Instruct-Reference"

# Set to True to load the model in 4-bit (saves ~12 GB VRAM, needs bitsandbytes)
USE_4BIT = False

# Set to True to load the model in 8-bit (saves ~8 GB VRAM, needs bitsandbytes)
USE_8BIT = False

# Generation parameters (mirrors the model's generation_config.json)
GENERATION_KWARGS = dict(
    max_new_tokens=512,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)

# ── Device ─────────────────────────────────────────────────────────────────────

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Quantization config ────────────────────────────────────────────────────────

bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    print("Loading model in 4-bit quantization...")
elif USE_8BIT:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    print("Loading model in 8-bit quantization...")
else:
    print("Loading model in bfloat16 (full precision)...")

# ── Load tokenizer ─────────────────────────────────────────────────────────────

print(f"\nLoading tokenizer from: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Llama 3 uses right-pad by convention for generation
tokenizer.padding_side = "right"

# ── Load model ─────────────────────────────────────────────────────────────────

print(f"Loading model from: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",          # spreads across all available GPUs/CPU automatically
    quantization_config=bnb_config,
)
model.eval()
print("Model loaded successfully.\n")

# ── Inference helper ───────────────────────────────────────────────────────────

def predict(user_message: str, system_prompt: str = None) -> str:
    """
    Run inference using the model's chat template.

    Args:
        user_message: The protein / gene description or query.
        system_prompt: Optional system-level instruction.

    Returns:
        The assistant's generated text.
    """
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": user_message})

    # Apply the chat template that ships with the model
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **GENERATION_KWARGS,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (skip the prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()


# ── Example usage ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    SYSTEM_PROMPT = (
        "You are an expert in Gene Ontology (GO). "
        "Given a protein or gene description, predict the relevant GO term IDs "
        "and provide a brief explanation for each."
    )

    # --- Single example ---
    query = (
        "The protein is involved in the phosphorylation of serine residues "
        "in target proteins during the cell cycle."
    )

    print("=" * 70)
    print("Query:")
    print(query)
    print("-" * 70)
    response = predict(query, system_prompt=SYSTEM_PROMPT)
    print("Response:")
    print(response)
    print("=" * 70)

    # --- Interactive loop ---
    print("\nEntering interactive mode. Type 'quit' or 'exit' to stop.\n")
    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nExiting.")
            break

        if user_input.lower() in {"quit", "exit", "q"}:
            print("Exiting.")
            break

        if not user_input:
            continue

        answer = predict(user_input, system_prompt=SYSTEM_PROMPT)
        print(f"\nModel: {answer}\n")


Using device: cuda
GPU: Tesla T4
VRAM available: 15.6 GB
Loading model in bfloat16 (full precision)...

Loading tokenizer from: suswitha/GO-Onotology-Term-IDMeta-Llama-3.1-8B-Instruct-Reference


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/897 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded successfully.

Query:
The protein is involved in the phosphorylation of serine residues in target proteins during the cell cycle.
----------------------------------------------------------------------
Response:
GO:0018105: Protein serine/threonine kinase activity 
GO:0004672: Protein kinase activity
GO:0008283: Cell proliferation 
GO:0051246: Regulation of cell cycle

Entering interactive mode. Type 'quit' or 'exit' to stop.

You: Endosome membrane?

Model: GO:0010008



In [ ]:
SYSTEM_PROMPT = (
        "You are an expert in Gene Ontology (GO). "
        "Given a protein or gene description, predict the relevant GO term IDs "
        "and provide a brief explanation for each."
    )

query = (
        "Endosome membrane? "
    )

response = predict(query, system_prompt=SYSTEM_PROMPT)
print(response)

GO:0010008


# Not an interactive loop

In [ ]:
"""
Inference script for:
  suswitha/GO-Onotology-Term-IDMeta-Llama-3.1-8B-Instruct-Reference

Fine-tuned Llama 3.1-8B-Instruct model for GO (Gene Ontology) Term ID prediction.

Requirements:
    pip install torch transformers accelerate bitsandbytes
"""

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# ── Configuration ──────────────────────────────────────────────────────────────

MODEL_ID = "suswitha/GO-Onotology-Term-IDMeta-Llama-3.1-8B-Instruct-Reference"

# Set to True to load the model in 4-bit (saves ~12 GB VRAM, needs bitsandbytes)
USE_4BIT = False

# Set to True to load the model in 8-bit (saves ~8 GB VRAM, needs bitsandbytes)
USE_8BIT = False

# Generation parameters (mirrors the model's generation_config.json)
GENERATION_KWARGS = dict(
    max_new_tokens=512,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
)

# ── Device ─────────────────────────────────────────────────────────────────────

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Quantization config ────────────────────────────────────────────────────────

bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    print("Loading model in 4-bit quantization...")
elif USE_8BIT:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    print("Loading model in 8-bit quantization...")
else:
    print("Loading model in bfloat16 (full precision)...")

# ── Load tokenizer ─────────────────────────────────────────────────────────────

print(f"\nLoading tokenizer from: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Llama 3 uses right-pad by convention for generation
tokenizer.padding_side = "right"

# ── Load model ─────────────────────────────────────────────────────────────────

print(f"Loading model from: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",          # spreads across all available GPUs/CPU automatically
    quantization_config=bnb_config,
)
model.eval()
print("Model loaded successfully.\n")

# ── Inference helper ───────────────────────────────────────────────────────────

def predict(user_message: str, system_prompt: str = None) -> str:
    """
    Run inference using the model's chat template.

    Args:
        user_message: The protein / gene description or query.
        system_prompt: Optional system-level instruction.

    Returns:
        The assistant's generated text.
    """
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": user_message})

    # Apply the chat template that ships with the model
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **GENERATION_KWARGS,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (skip the prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()


# ── Example usage ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    SYSTEM_PROMPT = (
        "You are an expert in Gene Ontology (GO). "
        "Given a protein or gene description, predict the relevant GO term IDs "
        "and provide a brief explanation for each."
    )

    query = (
        "The protein is involved in the phosphorylation of serine residues "
        "in target proteins during the cell cycle."
    )

    response = predict(query, system_prompt=SYSTEM_PROMPT)
    print(response)